In [38]:
import time
import random
import json
from pydantic import BaseModel
from dotenv import load_dotenv
from google import genai
from google.genai import types
from google.genai import errors
from unittest.mock import MagicMock
from dataclasses import dataclass, field

In [39]:
load_dotenv()
client = genai.Client()

### Gemini api call

In [40]:
@dataclass
class API_Configs:
    model_config = types.GenerateContentConfig(
    system_instruction="Respond STRICTLY with JSON format. Do not add markdown fences.",
    max_output_tokens=10240,
)
    
    model_tiers = {
    "high": {"model_name": "gemini-3.5-flash"},
    "medium": {"model_name": "gemini-2.5-flash"},
    "low": {"model_name": "gemini-3.1-flash-lite"}
}

In [41]:
@dataclass
class Retry_Config:
    max_attempts: int = 3
    base_delay: float = 1.0
    max_delay: float = 30.0

    retry_status_codes = (429, 500, 503, 504)

In [42]:
def execute_prompt(model_name, prompt):
    for attempt in range (1, Retry_Config.max_attempts + 1):

        # Api call Section
        try:
            if attempt > 1:
                print(f"Retry attempt {attempt}/{Retry_Config.max_attempts}...")
            else:
                print(f"Sending initial request...")

            stream = client.models.generate_content_stream(model = model_name, contents = prompt, config = API_Configs.model_config)
            
            #Consumes the streamed chunks during execution so that errors can be caught early instead of waiting for the output to be parsed.
            for chunk in stream:
                yield chunk

            return 
        
        # Error handling section
        except errors.APIError as e:

            # Refers to the Tuple of status codes that can be retried in Retry_Config.
            if e.code in Retry_Config.retry_status_codes:
                if attempt == Retry_Config.max_attempts:
                    print("Max attempts reached.")
                    raise e

                # Exponential backoff, i should probably add a way to check the server header for the exact time needed if it exists later (keyword: *later*)
                new_delay = random.uniform(0.5, 1.5) * Retry_Config.base_delay * (2 ** (attempt - 1))
                clamped_delay = min(new_delay, Retry_Config.max_delay)

                print(f" Code {e.code}. Backing off. Waiting {clamped_delay:.2f}s...")
                time.sleep(clamped_delay)
                continue

            else:
                status_reason = getattr(e, "status", "UNKNOWN")
                
                if e.code == 404 or status_reason == "NOT_FOUND": # Where is this error? i cant find it? Muhehehehehehhe
                    print(f"Model Not Found [404]: '{model_name}' does not exist or is deprecated.")
                    
                elif e.code in (401, 403) or status_reason == "PERMISSION_DENIED":
                    print(f"Authentication Failed [{e.code}]: Check your API key, project quotas, or permissions.")
                    
                elif e.code == 400 or status_reason == "INVALID_ARGUMENT":
                    print(f"Invalid Argument [400]: Malformed structure or prompt content limits exceeded.")
                    
                else:
                    print(f"Non-retryable error [{status_reason}] ({e.code}): {e.message}")
                    
                raise e

In [43]:
def get_structured_output(model_name, prompt):
    response_stream = execute_prompt(model_name = model_name, prompt = prompt, )

    full_response = "".join(getattr(chunk, "text", "") or "" for chunk in response_stream)

    data = json.loads(full_response)
    return data

In [44]:
model_priority = "medium"

In [45]:
tier = API_Configs.model_tiers.get(model_priority.lower(), API_Configs.model_tiers["medium"])
model_name = tier["model_name"]

Example Prompts:

-You shall henceforth and at once provide me with TEN of the greatest most glorified spaghettie recipes that have crossed your existence.

-Say hello in 10 random languages and state which language it is.

In [48]:
# === INTERACTIVE TERMINAL SIMULATOR CELL (WITH UI FLUSH FIX) ===
import sys
import time

print("====================================================")
print("      Gemini Interactive Chat Terminal")
print("  Type 'exit' or 'quit' to end the session.")
print("====================================================")
sys.stdout.flush() # Force initial render

while True:
    try:
        # 1. Grab user input
        user_prompt = input("\nEnter your prompt (e.g., Ask for recipes, translations, etc.):\n> ")
        
        if user_prompt.strip().lower() in ['exit', 'quit']:
            print("\n Terminal session closed.")
            sys.stdout.flush()
            break
            
        if not user_prompt.strip():
            continue

        print("\nThinking... Calling API...")
        sys.stdout.flush() # Force "Thinking..." to show up immediately

        # 2. Leverage your existing variables and functions completely untouched
        result_data = get_structured_output(model_name=model_name, prompt=user_prompt)
        
        # 3. Print the formatted structured output
        print("\n" + "="*50)
        print("RESPONSE:")
        print("="*50)
        print(json.dumps(result_data, indent=4, ensure_ascii=False))
        print("="*50)
        
        # 4. CRITICAL: Force Jupyter to flush the text stream to the screen
        sys.stdout.flush()
        
        # Give the Jupyter notebook front-end a 50ms breather to render the text
        time.sleep(0.05)

    except json.JSONDecodeError:
        print("\n Parsing Error: The model returned unstructured text or invalid JSON.")
        sys.stdout.flush()
    except Exception as e:
        print(f"\n Error occurred: {e}")
        sys.stdout.flush()

      Gemini Interactive Chat Terminal
  Type 'exit' or 'quit' to end the session.

Thinking... Calling API...
Sending initial request...

 Error occurred: [Errno 11001] getaddrinfo failed

Thinking... Calling API...
Sending initial request...

RESPONSE:
{
    "translations": [
        {
            "country": "United Kingdom",
            "language": "English",
            "translation": "We have lost objective butter."
        },
        {
            "country": "France",
            "language": "French",
            "translation": "Nous avons perdu le beurre objectif."
        },
        {
            "country": "Russian Empire",
            "language": "Russian",
            "translation": "Мы потеряли объективное масло."
        },
        {
            "country": "United States",
            "language": "English",
            "translation": "We have lost objective butter."
        },
        {
            "country": "Italy",
            "language": "Italian",
            "transla